In [ ]:
import traceback
from IPython.display import display, HTML

def show(html):
    display(HTML(html))

try:
    import subprocess, time, os, socket

    port = 3845
    path = '/home/jovyan/tools/collocationcalculator'
    name = 'CollocationCalculator'

    # Kill any stale process on this port
    try:
        subprocess.run(['fuser', '-k', str(port) + '/tcp'],
                       capture_output=True, check=False, timeout=5)
        time.sleep(1)
    except Exception:
        pass

    # Write R startup script
    r_script = '/tmp/start_collocationcalculator.R'
    with open(r_script, 'w') as f:
        f.write(
            "shiny::runApp('" + path + "', port=" + str(port) +
            ", host='0.0.0.0', launch.browser=FALSE)\n"
        )

    log_path = '/tmp/shiny_collocationcalculator.log'
    with open(log_path, 'w') as log:
        proc = subprocess.Popen(
            ['Rscript', '--vanilla', r_script],
            stdout=log, stderr=subprocess.STDOUT
        )

    # Poll until port opens or process exits (max 120 s)
    ready = False
    for _ in range(120):
        time.sleep(1)
        if proc.poll() is not None:
            break
        try:
            with socket.create_connection(('localhost', port), timeout=1):
                pass
            ready = True
            break
        except OSError:
            pass

    if ready:
        base_url = os.environ.get('JUPYTERHUB_SERVICE_PREFIX', '/')
        tool_url = base_url.rstrip('/') + '/proxy/' + str(port) + '/'
        show(
            '<iframe src="' + tool_url + '" '
            'width="100%" height="900" frameborder="0" '
            'style="border-radius:8px;border:1px solid #e0d8ec;"></iframe>'
        )
    else:
        try:
            with open(log_path) as f:
                log_txt = f.read()[-5000:]
        except Exception:
            log_txt = '(log not available)'
        show(
            '<div style="font-family:sans-serif;padding:16px 20px;background:#fff0f0;'
            'border-left:4px solid #e74c3c;border-radius:6px;">'
            '<b style="color:#c0392b;">&#x274C; ' + name + ' failed to start.</b><br><br>'
            '<b>Error log:</b><br>'
            '<pre style="background:#fff8f8;padding:10px;border-radius:4px;'
            'font-size:.78rem;white-space:pre-wrap;max-height:400px;'
            'overflow-y:auto;border:1px solid #fcc;">' + log_txt + '</pre></div>'
        )

except SystemExit:
    pass
except Exception:
    tb = traceback.format_exc()
    show(
        '<div style="font-family:monospace;padding:16px;background:#fff0f0;'
        'border-left:4px solid #e74c3c;border-radius:6px;">'
        '<b style="font-family:sans-serif;color:#c0392b;">'
        '&#x274C; Launcher error:</b><br><br>'
        '<pre style="font-size:.8rem;white-space:pre-wrap;">' + tb + '</pre></div>'
    )
